# 02b - Preprocessing of the full ESM25 dataset

Applies to every ESM25 record the preprocessing chosen on the development subset in notebook 02 (window, anchor,
resampling, spectrogram and scalogram on the common grid; parameters in the `preprocessing` section of
`configs/default.yaml`). No exploratory analysis is repeated here.

- Metadata and traces are read from the full SeisBench datasets in the cache, whatever `data.source` and
  `waveforms.source` say (so notebooks 01 and 02 can stay on the development subset).
- Output: `data/processed/seisbench/features.h5` and `records.csv`; to train on it, set
  `training.feature_source: seisbench`.
- The run is **resumable**: if it stops (kernel restart, sleep, error), run the notebook again and it continues from
  the records not yet processed. If the file is already complete, the preprocessing is skipped.
- At the end: checks on the stored records and comparison with the development subset (the same records must give the
  same features).

## 0. Setup

In [1]:
from dataclasses import replace
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

from wqu.config import load_config
from wqu.data.exploration import numeric_summary_by_label
from wqu.data.loading import CV_COMPONENTS, load_esm25_metadata
from wqu.data.waveforms import iter_waveform_batches, stored_sampling_rate_hz
from wqu.preprocessing.pipeline import (
    FEATURE_INDEX_COLUMN,
    REPRESENTATIONS,
    feature_file_path,
    grid_from_config,
    preprocess_and_store,
    read_feature_records,
    read_features,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
FULL_SOURCE = "seisbench"
DEV_SOURCE = "dev_subset"
BYTES_PER_VALUE = 2
COMPARISON_BLOCK = 250

# Metadata and traces from the SeisBench datasets, whatever data.source and waveforms.source say.
config = load_config(CONFIG_PATH)
config = replace(
    config,
    data=replace(config.data, source=FULL_SOURCE),
    waveforms=replace(config.waveforms, source=FULL_SOURCE),
)
columns = config.columns
preprocessing = config.preprocessing
label_names = {subset.label: subset.name for subset in config.data.subsets}
grid = grid_from_config(preprocessing)

_, metadata = load_esm25_metadata(config)
sampling_rate_hz = stored_sampling_rate_hz(metadata)
features_path = feature_file_path(preprocessing, FULL_SOURCE)
print(f"Records: {len(metadata):,}; window {preprocessing.pre_anchor_s:g} s before to {preprocessing.post_anchor_s:g} s "
      f"after the anchor ({preprocessing.anchor_method}); grid {len(grid.frequencies_hz)} x {len(grid.times_s)}")
print(f"Output: {features_path}")

/Users/giulianaparadiso/miniconda3/envs/wqu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Records: 100,249; window 30 s before to 90 s after the anchor (envelope); grid 128 x 128
Output: /Users/giulianaparadiso/Desktop/Projects/waveform-quality-uncertainty/data/processed/seisbench/features.h5


## 1. Disk space

Size of the feature file (both representations, float16) against the free space on the disk.

In [2]:
values_per_record = len(REPRESENTATIONS) * len(CV_COMPONENTS) * len(grid.frequencies_hz) * len(grid.times_s)
required_bytes = len(metadata) * values_per_record * BYTES_PER_VALUE
features_path.parent.mkdir(parents=True, exist_ok=True)
free_bytes = shutil.disk_usage(features_path.parent).free
print(f"Feature file: {required_bytes / 1e9:.1f} GB; free space: {free_bytes / 1e9:.1f} GB")
if not features_path.exists() and free_bytes < required_bytes:
    raise RuntimeError("Not enough free disk space for the feature file.")

Feature file: 19.7 GB; free space: 94.9 GB


## 2. Preprocessing (resumable)

About two hours on a laptop. The progress bar starts from the records already processed in a previous run.

In [3]:
if features_path.exists():
    stored_records = read_feature_records(features_path, columns.text)
    print(f"Feature file already complete: {len(stored_records):,} records")
else:
    stored_records = preprocess_and_store(
        metadata, lambda selected: iter_waveform_batches(selected, config), sampling_rate_hz, preprocessing,
        config.waveforms.zero_tolerance, features_path,
    )
    print(f"Stored {len(stored_records):,} records in {features_path}")

preprocessing: 100%|██████████| 100249/100249 [2:07:11<00:00, 13.14it/s] 


Stored 100,249 records in /Users/giulianaparadiso/Desktop/Projects/waveform-quality-uncertainty/data/processed/seisbench/features.h5


## 3. Checks on the stored records

Records per class, window statistics by class (to compare with notebook 02 on the development subset) and share of
records with particular windows.

In [4]:
print(stored_records[columns.label].map(label_names).value_counts().to_string())
WINDOW_COLUMNS = ["anchor_time_s", "recorded_fraction_kept", "energy_fraction_kept",
                  *[f"missing_in_window_s_{component}" for component in CV_COMPONENTS]]
numeric_summary_by_label(stored_records, WINDOW_COLUMNS, columns.label).rename(index=label_names, level=columns.label).round(3)

label
good    64213
bad     36036


count     mean     std    min     25%     50%  \
column                 label                                                    
anchor_time_s          good   64213.0   76.058  39.581  1.000  61.995  81.090   
                       bad    36036.0  105.414  72.616  0.000  71.996  89.960   
recorded_fraction_kept good   64213.0    0.534   0.236  0.061   0.401   0.433   
                       bad    36036.0    0.433   0.192  0.000   0.330   0.411   
energy_fraction_kept   good   64213.0    0.922   0.142  0.073   0.918   0.989   
                       bad    36036.0    0.649   0.245  0.000   0.449   0.640   
missing_in_window_s_u  good   64213.0   11.095  23.721  0.000   0.000   0.000   
                       bad    36036.0   13.805  27.031  0.000   0.000   0.000   
missing_in_window_s_v  good   64213.0   11.096  23.725  0.000   0.000   0.000   
                       bad    36036.0   13.927  27.268  0.000   0.000   0.000   
missing_in_window_s_w  good   64213.0   11.098  23.727  0.000   0.000   0.000   
                       bad    36036.0   14.019  27.425  0.000   0.000   0.000   

                                  75%      max  
column                 label                    
anchor_time_s          good    93.055  413.580  
                       bad    117.470  413.165  
recorded_fraction_kept good     0.587    1.000  
                       bad      0.436    1.000  
energy_fraction_kept   good     0.999    1.000  
                       bad      0.880    1.000  
missing_in_window_s_u  good     1.850  115.065  
                       bad     13.798  120.000  
missing_in_window_s_v  good     1.915  120.000  
                       bad     13.855  120.000  
missing_in_window_s_w  good     1.860  115.030  
                       bad     14.100  120.000

In [5]:
window_samples = stored_records["window_stop"] - stored_records["window_start"]
missing = stored_records[[f"missing_in_window_s_{component}" for component in CV_COMPONENTS]]
window_flags = pd.DataFrame(
    {
        "window starts before the recording": stored_records["window_start"] < 0,
        "some data missing in the window": missing.max(axis=1) > 0,
        "a component without data in the window": missing.max(axis=1) >= preprocessing.window_length_s,
        "energy kept < 95%": stored_records["energy_fraction_kept"] < 0.95,
    }
)
window_flags.groupby(stored_records[columns.label]).mean().T.rename(columns=label_names).round(3)

label,good,bad
window starts before the recording,0.173,0.126
some data missing in the window,0.255,0.316
a component without data in the window,0.000,0.009
energy kept < 95%,0.316,0.837


## 4. Comparison with the development subset

The records of the development subset are also in the full dataset: their windows and features must be the same in
the two files (the traces come from the same SeisBench datasets).

In [6]:
dev_path = feature_file_path(preprocessing, DEV_SOURCE)
dev_records = read_feature_records(dev_path, columns.text)
full_rows = stored_records.loc[dev_records.index, FEATURE_INDEX_COLUMN].to_numpy()
dev_rows = dev_records[FEATURE_INDEX_COLUMN].to_numpy()
same_window = (stored_records.loc[dev_records.index, "window_start"].to_numpy() == dev_records["window_start"].to_numpy())
print(f"Development records: {len(dev_records):,}; same window start: {same_window.mean():.4f}")

differences = {}
for representation in REPRESENTATIONS:
    largest = 0.0
    for start in range(0, len(dev_rows), COMPARISON_BLOCK):
        stop = start + COMPARISON_BLOCK
        dev_values, _ = read_features(dev_path, representation, dev_rows[start:stop])
        full_values, _ = read_features(features_path, representation, full_rows[start:stop])
        largest = max(largest, float(np.abs(dev_values - full_values).max()))
    differences[representation] = largest
pd.Series(differences, name="largest difference (dB)")

Development records: 2,000; same window start: 1.0000


spectrogram    0.0
scalogram      0.0
Name: largest difference (dB), dtype: float64